<a href="https://colab.research.google.com/github/ryanrtrindade/Aluguel_irlanda/blob/C%C3%B3digos/previsao_aluguel_irlanda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Análise e Avaliação do Modelo de Previsão de Preços de Imóveis


Objetivo

Este estudo tem como objetivo desenvolver e avaliar um modelo de machine learning capaz de prever o preço de imóveis com base em características como localização, número de quartos e período temporal.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


df = pd.read_csv('/content/drive/MyDrive/PYTHON/irish/irish_rent_full.csv')

In [2]:
df

,rent_euro,year,half,half_year,time_period,county,province,area,location,property_type,bedrooms,bedrooms_num,is_dublin,is_city,is_county_aggregate
0,835.90,2020,1,2020H1,1,Carlow,Leinster,Carlow,Carlow,All property types,All bedrooms,NaN,False,False,True
1,860.74,2020,1,2020H1,1,Carlow,Leinster,Carlow Town,Carlow Town,All property types,All bedrooms,NaN,False,False,False
2,910.91,2020,1,2020H1,1,Carlow,Leinster,Graiguecullen,"Graiguecullen, Carlow",All property types,All bedrooms,NaN,False,False,False
3,812.14,2020,1,2020H1,1,Carlow,Leinster,Tullow,"Tullow, Carlow",All property types,All bedrooms,NaN,False,False,False
4,666.31,2020,1,2020H1,1,Cavan,Ulster,Cavan,Cavan,All property types,All bedrooms,NaN,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50203,2090.44,2025,1,2025H1,11,Limerick,Munster,Limerick,Limerick,Terrace house,Four plus bed,4.0,False,False,True
50204,2363.79,2025,1,2025H1,11,Limerick,Munster,Limerick City,Limerick City,Terrace house,Four plus bed,4.0,False,True,False
50205,2343.70,2025,1,2025H1,11,Cork,Munster,Cork,Cork,Apartment,Four plus bed,4.0,False,False,True
50206,3025.10,2025,1,2025H1,11,Dublin,Leinster,Dublin,Dublin,Apartment,Four plus bed,4.0,True,True,True


A primeira etapa será entender os dados presentes no dataset e fazer o tratamento necessário em variáveis nulas.

In [3]:
df.dtypes

,0
rent_euro,float64
year,int64
half,int64
half_year,object
time_period,int64
county,object
province,object
area,object
location,object
property_type,object


In [4]:
df.isnull().sum()

,0
rent_euro,0
year,0
half,0
half_year,0
time_period,0
county,0
province,0
area,0
location,0
property_type,0


In [5]:
df['bedrooms_num'].value_counts()

,count
bedrooms_num,
2.0,17448
1.5,7708
3.0,6080
1.0,3600
4.0,3033


In [6]:
df['bedrooms_num'] = df['bedrooms_num'].fillna(df['bedrooms_num'].median())

In [7]:
df

,rent_euro,year,half,half_year,time_period,county,province,area,location,property_type,bedrooms,bedrooms_num,is_dublin,is_city,is_county_aggregate
0,835.90,2020,1,2020H1,1,Carlow,Leinster,Carlow,Carlow,All property types,All bedrooms,2.0,False,False,True
1,860.74,2020,1,2020H1,1,Carlow,Leinster,Carlow Town,Carlow Town,All property types,All bedrooms,2.0,False,False,False
2,910.91,2020,1,2020H1,1,Carlow,Leinster,Graiguecullen,"Graiguecullen, Carlow",All property types,All bedrooms,2.0,False,False,False
3,812.14,2020,1,2020H1,1,Carlow,Leinster,Tullow,"Tullow, Carlow",All property types,All bedrooms,2.0,False,False,False
4,666.31,2020,1,2020H1,1,Cavan,Ulster,Cavan,Cavan,All property types,All bedrooms,2.0,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50203,2090.44,2025,1,2025H1,11,Limerick,Munster,Limerick,Limerick,Terrace house,Four plus bed,4.0,False,False,True
50204,2363.79,2025,1,2025H1,11,Limerick,Munster,Limerick City,Limerick City,Terrace house,Four plus bed,4.0,False,True,False
50205,2343.70,2025,1,2025H1,11,Cork,Munster,Cork,Cork,Apartment,Four plus bed,4.0,False,False,True
50206,3025.10,2025,1,2025H1,11,Dublin,Leinster,Dublin,Dublin,Apartment,Four plus bed,4.0,True,True,True


Nesta etapa, serão identificadas as colunas categóricas do conjunto de dados, com o objetivo de prepará-las para o processo de modelagem.
A necessidade dessa etapa é pelo fato de modelos de trabalharem diretamente com dados textuais.

In [8]:
colunas = df.dtypes

categ_cols = []

for i in colunas.index:
    if colunas[i] == 'object' or colunas[i] == 'bool':
        categ_cols.append(i)


In [9]:
categ_cols

['half_year',
 'county',
 'province',
 'area',
 'location',
 'property_type',
 'bedrooms',
 'is_dublin',
 'is_city',
 'is_county_aggregate']

In [10]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()


for i in categ_cols:
    df[str(i) + '_encoded'] = le.fit_transform(df[i])
    df = df.drop(i, axis = 1)



df[str(i) + '_encoded'] = le.fit_transform(df['year'])



In [11]:
df

,rent_euro,year,half,time_period,bedrooms_num,half_year_encoded,county_encoded,province_encoded,area_encoded,location_encoded,property_type_encoded,bedrooms_encoded,is_dublin_encoded,is_city_encoded,is_county_aggregate_encoded
0,835.90,2020,1,1,2.0,0,0,1,66,70,0,2,0,0,0
1,860.74,2020,1,1,2.0,0,0,1,67,71,0,2,0,0,0
2,910.91,2020,1,1,2.0,0,0,1,201,210,0,2,0,0,0
3,812.14,2020,1,1,2.0,0,0,1,388,401,0,2,0,0,0
4,666.31,2020,1,1,2.0,0,1,3,89,93,0,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50203,2090.44,2025,1,11,4.0,10,12,2,249,259,5,3,0,0,5
50204,2363.79,2025,1,11,4.0,10,12,2,250,260,5,3,0,1,5
50205,2343.70,2025,1,11,4.0,10,3,2,122,127,1,3,0,0,5
50206,3025.10,2025,1,11,4.0,10,5,1,145,152,1,3,1,1,5


Uma breve análise interessante sobre o preço médio dos alugueis:

-O preço vem subindo a cada ano
-Isso pode estar associado ao cenário econômico mundial pós pandemia
-Demonstra que o mercado imobiliário vem se valorizando a cada ano e que ter uma casa própria ou alugar uma é grande vem se tornando algo mais custeoso e a se considerar para os jovens que sonham em morar sozinho e começar sua vida

A grande pergunta: Será que uma hora o aluguel irá começar a baratear?

In [12]:
import plotly.express as px
media_year = df.groupby('year')['rent_euro'].mean().reset_index()


fig = px.line(media_year, x = 'year', y = 'rent_euro', title = 'Média de preço por ano', markers=True)
fig.update_layout(
    xaxis_title='Ano',
    yaxis_title='Preço médio (euros)'

)
fig.show()

Outra grande pergunta que é sempre considerada: O quanto o número de quartos influencia no preço de um imóvel?

Para pessoas casadas que sonham em ter filhos, é uma das primeiras coisas que veêm em mente.

Abaixo podemos ver alguns pontos:

-O número de quartos aumenta tende a ter preços mais elevados entra 1 e 2, e de 2 a 4

-Porém entre 2 e 3, imóveis de 2 quarto atingem valores mais altos, implicando que não necessiaramente números de quartos vão influenciar o preço, mas também localidade do imóvel

In [13]:
fig = px.strip(
    df, x='bedrooms_num', y='rent_euro', title='Distribuição de preço por número de quartos', color = 'bedrooms_num'
)
fig.update_layout(xaxis_title='Número de quartos',yaxis_title='Preço (euros)')
fig.show()

Esta é a parte onde o iremos selecionar a váriavel dependente e as variáveis independentes para treinar o modelo.

O modelo para previsão de preço será a árvore de decisão, e os motivos são:

-Facilidade de interpretar resultados

-Não há necessidade de normalizar dados

-Lida bem com dados categóricos

In [14]:
from sklearn.model_selection import train_test_split

x = df.drop('rent_euro', axis = 1)
y = df['rent_euro']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, random_state = 42)

In [15]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import  r2_score

modelo = RandomForestRegressor(max_depth= 500, max_leaf_nodes= 500, random_state= 42)
modelo.fit(x_train, y_train)
y_pred_test = modelo.predict(x_test)
y_pred_train = modelo.predict(x_train)

print(r2_score(y_test, y_pred_test))
print(r2_score(y_train, y_pred_train))

0.89956374144677
0.9157791986348243


In [16]:
depth = [10, 50, 100, 500]
leaf = [20, 50, 100, 500]


for i,j in zip(depth, leaf):
    print('depht:',i)
    print('leaf:',j)
    modelo = RandomForestRegressor(max_depth= i, max_leaf_nodes= j, random_state= 42)
    modelo.fit(x_train, y_train)
    y_pred_test = modelo.predict(x_test)
    y_pred_train = modelo.predict(x_train)

    print(r2_score(y_test, y_pred_test))
    print(r2_score(y_train, y_pred_train))

depht: 10
leaf: 20
0.7394818153689451
0.7336316669301812
depht: 50
leaf: 50
0.7980281284969818
0.7983427936745313
depht: 100
leaf: 100
0.8312682150393823
0.8365256836751276
depht: 500
leaf: 500
0.89956374144677
0.9157791986348243


In [17]:
df['preco_previsto'] = modelo.predict(x).round(2)

df['diferenca'] = df['rent_euro'] - df['preco_previsto'].round(2)


In [18]:
df

,rent_euro,year,half,time_period,bedrooms_num,half_year_encoded,county_encoded,province_encoded,area_encoded,location_encoded,property_type_encoded,bedrooms_encoded,is_dublin_encoded,is_city_encoded,is_county_aggregate_encoded,preco_previsto,diferenca
0,835.90,2020,1,1,2.0,0,0,1,66,70,0,2,0,0,0,821.48,14.42
1,860.74,2020,1,1,2.0,0,0,1,67,71,0,2,0,0,0,821.48,39.26
2,910.91,2020,1,1,2.0,0,0,1,201,210,0,2,0,0,0,802.74,108.17
3,812.14,2020,1,1,2.0,0,0,1,388,401,0,2,0,0,0,813.94,-1.80
4,666.31,2020,1,1,2.0,0,1,3,89,93,0,2,0,0,0,643.65,22.66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50203,2090.44,2025,1,11,4.0,10,12,2,249,259,5,3,0,0,5,1925.26,165.18
50204,2363.79,2025,1,11,4.0,10,12,2,250,260,5,3,0,1,5,2171.59,192.20
50205,2343.70,2025,1,11,4.0,10,3,2,122,127,1,3,0,0,5,2038.07,305.63
50206,3025.10,2025,1,11,4.0,10,5,1,145,152,1,3,1,1,5,3197.75,-172.65


In [19]:
df['classificacao'] = df['diferenca'].apply( lambda x:
  'barato' if x <0 else
  'dentro do orcamento' if 0 <  x <100 else
  'caro'
  )

In [20]:
df

,rent_euro,year,half,time_period,bedrooms_num,half_year_encoded,county_encoded,province_encoded,area_encoded,location_encoded,property_type_encoded,bedrooms_encoded,is_dublin_encoded,is_city_encoded,is_county_aggregate_encoded,preco_previsto,diferenca,classificacao
0,835.90,2020,1,1,2.0,0,0,1,66,70,0,2,0,0,0,821.48,14.42,dentro do orcamento
1,860.74,2020,1,1,2.0,0,0,1,67,71,0,2,0,0,0,821.48,39.26,dentro do orcamento
2,910.91,2020,1,1,2.0,0,0,1,201,210,0,2,0,0,0,802.74,108.17,caro
3,812.14,2020,1,1,2.0,0,0,1,388,401,0,2,0,0,0,813.94,-1.80,barato
4,666.31,2020,1,1,2.0,0,1,3,89,93,0,2,0,0,0,643.65,22.66,dentro do orcamento
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50203,2090.44,2025,1,11,4.0,10,12,2,249,259,5,3,0,0,5,1925.26,165.18,caro
50204,2363.79,2025,1,11,4.0,10,12,2,250,260,5,3,0,1,5,2171.59,192.20,caro
50205,2343.70,2025,1,11,4.0,10,3,2,122,127,1,3,0,0,5,2038.07,305.63,caro
50206,3025.10,2025,1,11,4.0,10,5,1,145,152,1,3,1,1,5,3197.75,-172.65,barato


O gráfico abaixo faz uma comparação entre o preço previsto e real, o ponto principal mostrado é que os preços reais e previsto estão próximos, reforçando ainda mais argumento que o modelo consegue fazer previsões com uma boa precisão.

In [21]:
fig = px.scatter(
    df,
    x = 'preco_previsto',
    y = 'rent_euro',
    color = 'classificacao',
    title = 'Previsão x Real')
fig.update_layout(
    xaxis_title='Preço previsto',
    yaxis_title='Preço real')
fig.show()

In [22]:


fig = px.histogram(df, x='diferenca')

fig.add_vline(x=0, line_dash="dash", line_color="red")

fig.update_layout(
    xaxis_title='Diferença entre preço real e preço previsto',
    yaxis_title='Frequência'
)

fig.show()

O pico da montanha no gráfico está em 0, significando dois apontamentos:

1) Que o modelo não está enviezado

2) A maioria dos erros estão próximos de zero, mostrando maior parte das previsões tem erro pequeno

In [23]:
df['erro_percentual'] = df['diferenca'] / df['rent_euro'] * 100
print(df['erro_percentual'].abs().mean())

9.42582928916805


In [24]:
df['erro_percentual'] = df['diferenca'] / df['rent_euro'] * 100
print(df['erro_percentual'].mean())

-1.6418700255085745


O modelo possui erro médio absoluto de 9,43%, ou seja, ele indica boa capacidade de previsão.

Ele possui viés médio de 1,64%, onde sugere leve tendência em superestimar preços.